# Retrieve data using method and provider 

In [1]:
import yaml
from pynanomapper import aa
from pynanomapper import units
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
import measurement
from pynanomapper import client_solr
from pynanomapper import client_ambit
from pynanomapper import annotation

ModuleNotFoundError: No module named 'ipywidgets'

In [ ]:
print('Select enanoMapper aggregated search service:')
style = {'description_width': 'initial'}
config,config_servers, config_security, auth_object, msg = aa.parseOpenAPI3()    
service_widget = widgets.Dropdown(
    options=config_servers['url'],
    description='Service:',
    disabled=False,
    style=style
)
if config_security is None:
    service = interactive(aa.search_service_open,url=service_widget)
else:
    print(msg)
    apikey_widget=widgets.Text(
            placeholder='',
            description=config_security,
            disabled=False,
            style=style
    )    
    service = interactive(aa.search_service_protected,url=service_widget,apikey=apikey_widget)    

display(service)

Select enanoMapper aggregated search service:
Enter `X-Gravitee-Api-Key` you have received upon subscription to http://api.ideaconsult.net


d:\nina\src\pynanomapper\pynanomapper\aa.py:54: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead.
  config_servers = json_normalize(config['servers'], None, ['url'])
d:\nina\src\pynanomapper\pynanomapper\aa.py:56: FutureWarning: pandas.io.json.json_normalize is deprecated, use pandas.json_normalize instead.
  config_security = json_normalize(config['components']['securitySchemes'], None, [])


interactive(children=(Dropdown(description='Service:', options=('https://api.ideaconsult.net/enanomapper', 'ht…

In [3]:
service_uri=service_widget.value
if auth_object!=None:
    auth_object.setKey(apikey_widget.value)
print("Sending queries to {}".format(service_uri))

Sending queries to https://api.ideaconsult.net/riskgone


In [4]:
facets = client_solr.Facets()

project_filter = "owner_name_s:NANoREG"
df = facets.summary(service_uri,auth_object, query=project_filter,fields=["topcategory_s","endpointcategory_s"])    
df.head()
#df['owner_name_s'].unique()


,topcategory_s,endpointcategory_s,Number of data points,endpointcategory_term,endpointcategory_name
0,TOX,ENM_0000068_SECTION,10990,http://www.bioassayontology.org/bao#ENM_0000068,Cell Viability
1,TOX,TO_GENETIC_IN_VITRO_SECTION,7811,http://www.bioassayontology.org/bao#BAO_0002167,Genetic toxicity invitro
2,TOX,NPO_1339_SECTION,3429,http://purl.obolibrary.org/obo/NPO_1339,Immunotoxicity
3,TOX,TO_REPEATED_ORAL_SECTION,2045,http://purl.enanomapper.org/onto/ENM_0000021,Repeated dose toxicity-oral
4,TOX,ENM_0000044_SECTION,1322,http://purl.enanomapper.org/onto/ENM_0000044,Barrier integrity


In [5]:
top_widget = widgets.Dropdown(
    options=df['topcategory_s'].unique(),
    value="P-CHEM",
    description='Select:',
    disabled=False,
)
category_widget = widgets.Dropdown(
    options=list(df[df['topcategory_s']=="P-CHEM"][["endpointcategory_name","endpointcategory_s"]].itertuples(index=False,name=None)),
    #value=,
    description='Category:',
    layout=Layout(width='50%'),
    disabled=False,
)
owner_widget = widgets.Dropdown(
    options=[],
    description='Select:',
    disabled=False,
)
method_widget = widgets.Dropdown(
    options=[],
    description='Select:',
    disabled=False,
)
def define_query(_top,_section,_owner,_method):
    #category_widget.options=df[df['topcategory_s']==top]['endpointcategory_s'].unique()
    filtered = df[df['topcategory_s']==_top]
    category_widget.options = list(filtered[["endpointcategory_name","endpointcategory_s"]].itertuples(index=False,name=None))

    df1 = facets.summary(service_uri,auth_object, query="{} AND endpointcategory_s:{}".format(project_filter,_section),
                        fields=["reference_owner_s"])    
    owner_widget.options = df1['reference_owner_s'].unique()
    df2 = facets.summary(service_uri,auth_object, query="{} AND endpointcategory_s:{} AND reference_owner_s:{}".format(project_filter,_section,_owner),
                        fields=["E.method_s"])        
    method_widget.options = df2['E.method_s'].unique()
    top = _top
    section= _section
    
    
interact(define_query,_top= top_widget,_section=category_widget,_owner=owner_widget,_method=method_widget)

interactive(children=(Dropdown(description='Select:', index=1, options=('TOX', 'P-CHEM', 'ECOTOX'), value='P-C…

<function __main__.define_query(_top, _section, _owner, _method)>

In [6]:
top = top_widget.value
section = category_widget.value
provider=owner_widget.value
method=method_widget.value.replace(" ","\ ")

query="topcategory_s:{} AND endpointcategory_s:{} AND reference_owner_s:{} AND E.method_s:{}".format(top_widget.value,category_widget.value,owner_widget.value,method)

docs_query = client_solr.StudyDocuments()
docs_query.settings['endpointfilter'] = None
docs_query.settings['query_guidance'] = None
docs_query.settings['query_organism'] = None
docs_query.setStudyFilter({' topcategory_s': top, 'endpointcategory_s':section})
#,'E.method_s' : method})
#,'reference_owner_s' : provider}) 
docs_query.settings['fields'] = "*"                    
query = docs_query.getQuery(textfilter=query,facets=None,fq=None, rows=10, _params=True, _conditions=True, _composition=False );
print(query)

{'q': '{!parent which=type_s:substance}(topcategory_s:TOX AND endpointcategory_s:ENM_0000068_SECTION AND reference_owner_s:UIB AND E.method_s:Impedance\\ adherent\\ cells)', 'fq': None, 'wt': 'json', 'fl': '*,[child parentFilter=filter(type_s:substance) childFilter="filter(type_s:study AND     topcategory_s:TOX AND endpointcategory_s:ENM_0000068_SECTION )  OR filter(type_s:params AND     topcategory_s:TOX AND endpointcategory_s:ENM_0000068_SECTION)  OR filter(type_s:conditions AND     topcategory_s:TOX AND endpointcategory_s:ENM_0000068_SECTION) " limit=10000]', 'json.facet': '', 'rows': 10}


In [7]:
r = client_solr.get(service_uri,query=query,auth=auth_object)
print(r.status_code)
docs=r.json()['response']['docs']
rows = docs_query.parse(docs,process=None)


200


In [8]:
#rows
import pandas as pd
results = pd.DataFrame(rows)
results.head()


,db,m.substance.name,m.public.name,m.materialprovider,m.substance.type,p.oht.module,p.oht.section,p.guidance,p.reference,p.reference_year,...,x.params.E.incubation_time_before_exposure_UNIT,x.params.E.incubation_time_before_exposure_d,x.params.E.post_exposure_time_UNIT,x.params.E.post_exposure_time_d,x.params.E.supports,x.conditions.alternative_exposure_concentration2_UNIT,x.conditions.alternative_exposure_concentration2_d,x.conditions.alternative_exposure_concentration_UNIT,x.conditions.alternative_exposure_concentration_d,x.params.E.cell_source
0,NNRG,NM-100 TiO2 110 nm,JRCNM01000a,NANoREG,CHEBI_51050,TOX,ENM_0000068_SECTION,IMPEDANCE ADHERENT CELLS,UiB-NM100-1st,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NNRG,NM-100 TiO2 110 nm,JRCNM01000a,NANoREG,CHEBI_51050,TOX,ENM_0000068_SECTION,IMPEDANCE ADHERENT CELLS,UiB-NM100-1st,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NNRG,NM-100 TiO2 110 nm,JRCNM01000a,NANoREG,CHEBI_51050,TOX,ENM_0000068_SECTION,IMPEDANCE ADHERENT CELLS,UiB-NM100-1st,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NNRG,NM-100 TiO2 110 nm,JRCNM01000a,NANoREG,CHEBI_51050,TOX,ENM_0000068_SECTION,IMPEDANCE ADHERENT CELLS,UiB-NM100-1st,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NNRG,NM-100 TiO2 110 nm,JRCNM01000a,NANoREG,CHEBI_51050,TOX,ENM_0000068_SECTION,IMPEDANCE ADHERENT CELLS,UiB-NM100-1st,2015,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
results.loc[results["x.params.E.method"]==method_widget.value].to_excel("results.xlsx", index=False)

